# Baseline semantic search: TF-IDF và BM25

Notebook này xây dựng baseline tìm kiếm phim từ metadata TMDB và dữ liệu MovieLens.

- `LANGUAGE_MODE = "en"` dùng cột tiếng Anh nếu có, sau đó fallback về cột chung.
- Đổi sang `LANGUAGE_MODE = "vi"` khi metadata tiếng Việt đã sẵn sàng.
- TF-IDF và BM25 dùng chung dataframe, tokenizer và hàm `search()`.
- Phần load dữ liệu độc lập với model để có thể thay bằng embedding hoặc collaborative filtering sau này.

In [6]:
from __future__ import annotations

import json
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

In [7]:
DATA_DIR = Path("../data")
MOVIELENS_DIR = DATA_DIR / 'movielens'
TMDB_DIR = DATA_DIR / 'tmdb'
ARTIFACT_DIR = DATA_DIR / 'artifacts' / 'lexical_search'

LANGUAGE_MODE = 'vi'  # 'en' hoặc 'vi'
TOP_K = 10
MIN_OVERVIEW_CHARS = 20

if LANGUAGE_MODE not in {'en', 'vi'}:
    raise ValueError("LANGUAGE_MODE phải là 'en' hoặc 'vi'")

print(f'Data directory: {DATA_DIR}')
print(f'Language mode: {LANGUAGE_MODE}')

Data directory: ..\data
Language mode: vi


## 1. Load và chuẩn hóa dữ liệu

TMDB được lưu thành nhiều file JSONL theo batch, vì vậy loader dùng glob và loại bản ghi trùng `movieId`. MovieLens chỉ được đọc những bảng cần cho baseline: `movies.csv`, `links.csv`, `ratings.csv`.

In [8]:
def load_movielens(data_dir: Path) -> dict[str, pd.DataFrame]:
    required = ['movies.csv', 'links.csv', 'ratings.csv']
    missing = [name for name in required if not (data_dir / name).exists()]
    if missing:
        raise FileNotFoundError(f'Thiếu file MovieLens: {missing}')

    return {
        'movies': pd.read_csv(data_dir / 'movies.csv'),
        'links': pd.read_csv(data_dir / 'links.csv'),
        'ratings': pd.read_csv(data_dir / 'ratings.csv'),
    }


def _flatten_genres(value: Any) -> str:
    if isinstance(value, list):
        return ' '.join(
            item.get('name', '') if isinstance(item, dict) else str(item)
            for item in value
        ).strip()
    if pd.isna(value) if not isinstance(value, (list, dict)) else False:
        return ''
    return str(value).replace('|', ' ')


def load_tmdb_metadata(tmdb_dir: Path) -> pd.DataFrame:
    metadata = pd.read_csv(TMDB_DIR / 'movies_data_fully_translated.csv', encoding='utf-8-sig')
    metadata.info() 
    
    if 'movieId' not in metadata.columns:
        raise KeyError('TMDB metadata phải có cột movieId để join với MovieLens')

    metadata['movieId'] = pd.to_numeric(metadata['movieId'], errors='coerce')
    metadata = metadata.dropna(subset=['movieId']).drop_duplicates('movieId')
    metadata['movieId'] = metadata['movieId'].astype('int64')
    if 'genres' in metadata:
        metadata['genres_text'] = metadata['genres'].map(_flatten_genres)
    else:
        metadata['genres_text'] = ''
    return metadata


def build_movie_catalog(
    movielens: dict[str, pd.DataFrame],
    tmdb_metadata: pd.DataFrame,
    language_mode: str,
) -> pd.DataFrame:
    if language_mode not in {'en', 'vi'}:
        raise ValueError("language_mode phải là 'en' hoặc 'vi'")

    ratings = movielens['ratings']
    rating_stats = ratings.groupby('movieId', as_index=False).agg(
        rating_count=('rating', 'count'),
        rating_mean=('rating', 'mean'),
    )
    catalog = (
        movielens['movies']
        .merge(movielens['links'], on='movieId', how='left')
        .merge(rating_stats, on='movieId', how='left')
        .merge(tmdb_metadata, on='movieId', how='inner', suffixes=('', '_tmdb'))
    )

    overview_candidates = (['overview_en', 'overview'] if language_mode == 'en'
                          else ['overview_vi', 'overview'])
    title_candidates = (['original_title', 'title'] if language_mode == 'en'
                        else ['title_vi', 'title'])

    def first_available(row: pd.Series, candidates: list[str]) -> str:
        for column in candidates:
            if column in row.index and pd.notna(row[column]) and str(row[column]).strip():
                return str(row[column]).strip()
        return ''

    catalog['search_title'] = catalog.apply(lambda row: first_available(row, title_candidates), axis=1)
    catalog['search_overview'] = catalog.apply(lambda row: first_available(row, overview_candidates), axis=1)
    catalog['search_text'] = (
        catalog['search_title'].fillna('') + ' ' +
        catalog['search_overview'].fillna('') + ' ' +
        catalog['genres_text'].fillna('')
    ).str.replace(r'\s+', ' ', regex=True).str.strip()
    catalog = catalog[catalog['search_overview'].str.len() >= MIN_OVERVIEW_CHARS].copy()
    return catalog.reset_index(drop=True)


movielens = load_movielens(MOVIELENS_DIR)
tmdb_metadata = load_tmdb_metadata(TMDB_DIR)
catalog = build_movie_catalog(movielens, tmdb_metadata, LANGUAGE_MODE)
print(f'MovieLens movies: {len(movielens["movies"]):,}')
print(f'TMDB metadata rows: {len(tmdb_metadata):,}')
print(f'Catalog rows with usable overview: {len(catalog):,}')
display(catalog[['movieId', 'search_title', 'search_overview', 'rating_count', 'rating_mean']].head(3))

<class 'pandas.DataFrame'>
RangeIndex: 9811 entries, 0 to 9810
Data columns (total 32 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   tmdbId                 9811 non-null   float64
 1   title_vi               9811 non-null   str    
 2   overview_vi            9811 non-null   str    
 3   genres                 9811 non-null   str    
 4   release_date           9811 non-null   str    
 5   poster_path            9800 non-null   str    
 6   adult                  9811 non-null   bool   
 7   backdrop_path          9558 non-null   str    
 8   belongs_to_collection  1616 non-null   str    
 9   budget                 9811 non-null   int64  
 10  homepage               1 non-null      str    
 11  id                     9811 non-null   int64  
 12  imdb_id                9810 non-null   str    
 13  origin_country         9811 non-null   str    
 14  original_language      9811 non-null   str    
 15  original_title 

,movieId,search_title,search_overview,rating_count,rating_mean
0,723,Due amici,"Bạn bè hai tuổi, Nunzio và Pino, có chung một ...",39.0,3.307692
1,895,Wenecja,Một câu chuyện sắp tới kể về một cậu bé tên là...,67.0,2.977612
2,1115,The Sleepover,"Thị trấn Derry có một bí mật, nhưng không ai n...",44.0,2.954545


## 2. Tokenizer và các model lexical

BM25 được cài đặt tối giản ngay trong notebook để giảm dependency và giữ API tương thích với model khác. Khi production hóa, lớp này có thể chuyển sang `rank_bm25` mà không đổi phần catalog hay hàm tìm kiếm.

In [9]:
TOKEN_PATTERN = re.compile(r"(?u)\b\w+\b")


def tokenize(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(str(text).lower())


class TfidfSearch:
    def __init__(self) -> None:
        self.vectorizer = TfidfVectorizer(
            tokenizer=tokenize,
            preprocessor=None,
            token_pattern=None,
            sublinear_tf=True,
            min_df=2,
        )
        self.matrix = None

    def fit(self, texts: Iterable[str]) -> 'TfidfSearch':
        self.matrix = self.vectorizer.fit_transform(texts)
        return self

    def scores(self, query: str) -> np.ndarray:
        if self.matrix is None:
            raise RuntimeError('Model chưa được fit')
        query_vector = self.vectorizer.transform([query])
        return linear_kernel(query_vector, self.matrix).ravel()


class Bm25Search:
    def __init__(self, k1: float = 1.5, b: float = 0.75) -> None:
        self.k1 = k1
        self.b = b
        self.documents: list[list[str]] = []
        self.idf: dict[str, float] = {}
        self.doc_lengths = np.array([], dtype=float)
        self.average_doc_length = 0.0

    def fit(self, texts: Iterable[str]) -> 'Bm25Search':
        self.documents = [tokenize(text) for text in texts]
        self.doc_lengths = np.array([len(doc) for doc in self.documents], dtype=float)
        self.average_doc_length = float(self.doc_lengths.mean()) if len(self.doc_lengths) else 0.0
        document_frequency: dict[str, int] = {}
        for document in self.documents:
            for token in set(document):
                document_frequency[token] = document_frequency.get(token, 0) + 1
        document_count = len(self.documents)
        self.idf = {
            token: np.log(1 + (document_count - frequency + 0.5) / (frequency + 0.5))
            for token, frequency in document_frequency.items()
        }
        return self

    def scores(self, query: str) -> np.ndarray:
        if not self.documents:
            raise RuntimeError('Model chưa được fit')
        query_tokens = tokenize(query)
        scores = np.zeros(len(self.documents), dtype=float)
        for index, document in enumerate(self.documents):
            term_counts = pd.Series(document).value_counts()
            denominator_length = self.k1 * (
                1 - self.b + self.b * len(document) / max(self.average_doc_length, 1e-12)
            )
            for token in query_tokens:
                frequency = term_counts.get(token, 0)
                if frequency:
                    scores[index] += self.idf.get(token, 0.0) * (
                        frequency * (self.k1 + 1) / (frequency + denominator_length)
                    )
        return scores

In [10]:
tfidf_model = TfidfSearch().fit(catalog['search_text'])
bm25_model = Bm25Search().fit(catalog['search_text'])


def search(query: str, model_name: str = 'tfidf', top_k: int = TOP_K) -> pd.DataFrame:
    models = {'tfidf': tfidf_model, 'bm25': bm25_model}
    if model_name not in models:
        raise ValueError(f'model_name phải thuộc {sorted(models)}')
    scores = models[model_name].scores(query)
    result = catalog.copy()
    result['score'] = scores
    columns = ['movieId', 'search_title', 'search_overview', 'genres_text', 'rating_count', 'rating_mean', 'score']
    return result.nlargest(top_k, 'score')[columns].reset_index(drop=True)


QUERY = 'space adventure friendship'
display(search(QUERY, model_name='tfidf'))
display(search(QUERY, model_name='bm25'))

,movieId,search_title,search_overview,genres_text,rating_count,rating_mean,score
0,107130,An Adventure in Space and Time,Một nhà diễn thuyết đạo diễn William Hartrell ...,"[{'id': 18, 'name': 'Phim Chính Kịch'}]",90.0,3.877778,0.208119
1,78251,Friendship!,Một người đàn ông trẻ từ Đông Đức đi đến San F...,"[{'id': 35, 'name': 'Phim Hài'}, {'id': 18, 'n...",46.0,3.217391,0.206381
2,60649,Space Chimps,"Khi một tàu vũ trụ bay trên một hành tinh, một...","[{'id': 16, 'name': 'Phim Hoạt Hình'}, {'id': ...",161.0,2.183230,0.151236
3,102989,The Color of Friendship,Maree Bok sống ở một trang trại ở Nam Phi. cha...,"[{'id': 18, 'name': 'Phim Chính Kịch'}]",17.0,3.764706,0.149593
4,101763,My Awkward Sexual Adventure,Một lập trình viên quá mệt mỏi và smlubby kế t...,"[{'id': 35, 'name': 'Phim Hài'}, {'id': 10749,...",46.0,3.097826,0.141798
5,92214,The Poseidon Adventure,Một con tàu du lịch đầu hàng trước một hành độ...,"[{'id': 12, 'name': 'Phim Phiêu Lưu'}, {'id': ...",64.0,2.539062,0.140540
6,115840,Charlotte's Web 2: Wilbur's Great Adventure,Con lợn biết tình bạn quan trọng như thế nào -...,"[{'id': 12, 'name': 'Phim Phiêu Lưu'}, {'id': ...",10.0,2.750000,0.136336
7,94114,Sharpay's Fabulous Adventure,Sau khi một người có tài năng nhìn thấy cô ấy ...,"[{'id': 10751, 'name': 'Phim Gia Đình'}, {'id'...",16.0,2.593750,0.127085
8,27456,Shackleton's Antarctic Adventure,Bí mật của Shaleton là một bộ phim khổng lồ nó...,"[{'id': 99, 'name': 'Phim Tài Liệu'}]",107.0,3.584112,0.125975
9,113218,Space Milkshake,Các phi hành gia bốn tầng bị mắc kẹt lại với n...,"[{'id': 878, 'name': 'Phim Khoa Học Viễn Tưởng...",13.0,2.884615,0.122856


,movieId,search_title,search_overview,genres_text,rating_count,rating_mean,score
0,107130,An Adventure in Space and Time,Một nhà diễn thuyết đạo diễn William Hartrell ...,"[{'id': 18, 'name': 'Phim Chính Kịch'}]",90.0,3.877778,12.218362
1,78251,Friendship!,Một người đàn ông trẻ từ Đông Đức đi đến San F...,"[{'id': 35, 'name': 'Phim Hài'}, {'id': 18, 'n...",46.0,3.217391,8.825436
2,94114,Sharpay's Fabulous Adventure,Sau khi một người có tài năng nhìn thấy cô ấy ...,"[{'id': 10751, 'name': 'Phim Gia Đình'}, {'id'...",16.0,2.593750,8.162232
3,115840,Charlotte's Web 2: Wilbur's Great Adventure,Con lợn biết tình bạn quan trọng như thế nào -...,"[{'id': 12, 'name': 'Phim Phiêu Lưu'}, {'id': ...",10.0,2.750000,7.969700
4,60649,Space Chimps,"Khi một tàu vũ trụ bay trên một hành tinh, một...","[{'id': 16, 'name': 'Phim Hoạt Hình'}, {'id': ...",161.0,2.183230,7.707182
5,101763,My Awkward Sexual Adventure,Một lập trình viên quá mệt mỏi và smlubby kế t...,"[{'id': 35, 'name': 'Phim Hài'}, {'id': 10749,...",46.0,3.097826,7.525893
6,102989,The Color of Friendship,Maree Bok sống ở một trang trại ở Nam Phi. cha...,"[{'id': 18, 'name': 'Phim Chính Kịch'}]",17.0,3.764706,7.507419
7,92214,The Poseidon Adventure,Một con tàu du lịch đầu hàng trước một hành độ...,"[{'id': 12, 'name': 'Phim Phiêu Lưu'}, {'id': ...",64.0,2.539062,7.361908
8,27456,Shackleton's Antarctic Adventure,Bí mật của Shaleton là một bộ phim khổng lồ nó...,"[{'id': 99, 'name': 'Phim Tài Liệu'}]",107.0,3.584112,7.166711
9,113218,Space Milkshake,Các phi hành gia bốn tầng bị mắc kẹt lại với n...,"[{'id': 878, 'name': 'Phim Khoa Học Viễn Tưởng...",13.0,2.884615,6.857909


## 3. Đánh giá nhanh và lưu artifact

Đây là kiểm tra sanity cho baseline, chưa phải đánh giá offline đầy đủ. Khi có tập query/ground truth, có thể thêm Precision@K, Recall@K, MRR hoặc NDCG mà không thay đổi model interface.

In [13]:
sample_queries = [
    'cuộc phiêu lưu',
    'hài hước lãng mạn',
    'thám tử',
]

for query in sample_queries:
    print(f'\nQuery: {query}')
    print(search(query, model_name='tfidf', top_k=5)[['movieId', 'search_title', 'search_overview', 'score']].to_string(index=False))


Query: cuộc phiêu lưu
 movieId                             search_title                                                                                                                                                  search_overview    score
  135502                       Cậu Bé Và Thế Giới                                                                                                          Câu chuyện về cuộc phiêu lưu đi tìm cha của một cậu bé. 0.414069
   79895 Những Cuộc Phiêu Lưu Của Adèle Blanc-Sec Một cuộc phiêu lưu diễn ra vào đầu thế kỷ 20, tập trung vào một phong cách phổ biến và giao dịch của cô với những kẻ xấu, cảnh sát, quái vật, và những thứ khác. 0.372196
   65685                      Trang Sách Ma Thuật               Những cuộc phiêu lưu của một người cha và đứa con gái nhỏ, trong cuộc tìm kiếm một cuốn sách đã mất từ lâu sẽ giúp cho một người thân bị mất tích. 0.361133
   42004                             Transamerica     Một người phụ nữ khác giới trải qua một cuộ

In [12]:
# Lưu catalog và cấu hình để API/service có thể dùng lại sau này.
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
catalog.to_parquet(ARTIFACT_DIR / f'catalog_{LANGUAGE_MODE}.parquet', index=False)
with (ARTIFACT_DIR / f'config_{LANGUAGE_MODE}.json').open('w', encoding='utf-8') as file:
    json.dump(
        {
            'language_mode': LANGUAGE_MODE,
            'text_columns': ['search_title', 'search_overview', 'genres_text'],
            'models': ['tfidf', 'bm25'],
            'tfidf': {'sublinear_tf': True, 'min_df': 2},
            'bm25': {'k1': bm25_model.k1, 'b': bm25_model.b},
            'document_count': len(catalog),
        },
        file,
        ensure_ascii=False,
        indent=2,
    )
print(f'Đã lưu artifacts tại: {ARTIFACT_DIR}')

Đã lưu artifacts tại: ..\data\artifacts\lexical_search
